# CNN-based classification of detected regions

This recipe goes over all region detected in an image (e.g. by the previous Cellpose notebook) and runs classification on each of them using a ResNet18 or a custom simple CNN model.

## Inputs

- path to folder containing the raw images to be classified (plus pattern for which images to include, e.g. `*_ch1*.tif` to run on channel 1 images)
- path to folder containing label maps for the images (resulting file lists need to match!)
- a **model info** file (pointing to the model weights)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torchvision import models
from torchvision.transforms import v2
from skimage.io import imread
from skimage.measure import regionprops
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path

from img_utils import convert_8bit_mask_background
from model_simplecnn import SimpleCNNClassifierLightning


DEFAULT_CLASS_COLORS = {
    'EarlyS': (255, 0, 0),
    'G1G2': (0, 255, 0),
    'LateS': (0, 0, 255),
    'MidS': (255, 255, 0),
    'ambiguous': (255, 255, 255)
}

def create_result_table(classes):
    return {
        "cell_id": [],
        "max_class": [],
        "max_class_name": [],
        "img_file": [],
        "mask_file": [],
    } | {f'prob_{cls}': []  for cls in classes}

# Parameters

In [ ]:
# input paths: base in_path
in_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/'

# subdirectory for tiff files (with pattern)
image_subdirectory = 'images/2603011/tif_IllCorr' # for resaved: "tif"
image_file_pattern = '*_ch1*.tif' # e.g. "*_ch1.tif" to only include images of one channel

mask_subdirectory = 'results/segmentation_nuclei_IllCorr'
mask_file_patterns = '*.tif'

results_subdirectory = 'cellcycle_classification_retrained'
visualization_subdirectory = 'vis'

model_info_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/models/2ndretrained_resnet18/resnet18_cellcycle_retrained.json'
# model_info_path = '/Users/david/Desktop/project_cellcycle_classification_demo_frcnn/resnet18_cellcycle.json'

## Load model, move to GPU if possible

In [ ]:
with open(model_info_path) as fd:
    model_info = json.load(fd)

# weights as relative path
weigths_path = Path(model_info_path).parent / model_info['weights_file']

# check model architecture
SUPPORTED_ARCHS = ['resnet18', 'simple_cnn']
if model_info['architecture'] not in SUPPORTED_ARCHS:
    raise ValueError(f'Only architectures {SUPPORTED_ARCHS} supported at the moment, was given {model_info["architecture"]}')

# --- Load Model ---
# NOTE: if model contains batch norm, don't forget to set to eval!
if model_info['architecture'] == 'resnet18':
    model = models.resnet18(num_classes=len(model_info['classes']))

    weights = torch.load(weigths_path, map_location='cpu')

    # Training script uses wrapper class that adds "model." prefix to all keys
    # remove since we now use a plain ResNet
    if all([k.startswith("model.") for k in weights]):
        weights = {k[6:]: v for k, v in weights.items()}
    
    model.load_state_dict(weights)
    model.eval()

elif model_info['architecture'] == 'simple_cnn':
    model = SimpleCNNClassifierLightning.load_from_checkpoint(weigths_path).eval()

# 
inference_transform = v2.Compose([
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Resize(model_info['input_shape']),
])

model_info

In [ ]:
# get accelerator device and move model there

if torch.cuda.is_available():
    device = torch.device('cuda:0')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    
model.to(device)

device

## Get input

In [ ]:
image_files = sorted((Path(in_path) / image_subdirectory).glob(image_file_pattern))
mask_files = sorted((Path(in_path) / mask_subdirectory).glob(mask_file_patterns))

# number of files for verification
len(image_files), len(mask_files)

## Make output folder and Run

In [ ]:
out_dir = Path(in_path) / results_subdirectory
visualization_dir = out_dir / visualization_subdirectory

if not out_dir.exists():
    out_dir.mkdir()
if not visualization_dir.exists():
    visualization_dir.mkdir()

In [ ]:
from skimage.morphology import dilation
from calmutils.morphology.structuring_elements import hypersphere_centered


for img_path, mask_path in zip(image_files, mask_files):

    # output file paths 
    annotated_img_path = visualization_dir / (img_path.stem + '.png')
    csv_path = out_dir / (img_path.stem + '_classification_results.csv')
    
    # Load Image and Mask, Expand Mask
    mask = imread(mask_path)
    img = imread(img_path)
    mask = dilation(mask, hypersphere_centered(mask.ndim, 5))    
    
    # --- Classification ---
    class_colors = DEFAULT_CLASS_COLORS
    table_dict = create_result_table(model_info['classes'])
    
    # 8bit image for classification visualization
    # and as normalized input to classifier
    img_8bit = convert_8bit_mask_background(img, mask)   
    annotated_img = cv2.cvtColor(img_8bit, cv2.COLOR_GRAY2BGR)
    
    # TODO: get patches first, classify in batch?
    # seems fast enough as-is at the moment
    for rprop in regionprops(mask):
        img_patch = img_8bit[rprop.slice]

        # repeat image to match input shape
        if img_patch.ndim == 2:
            img_patch = np.stack([img_patch]*model_info['input_channels'], axis=-1)

        # apply transformations and run network
        input_tensor = inference_transform(img_patch).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
    
        class_idx = np.argmax(probs)
        class_name = model_info['classes'][class_idx]
    
        table_dict['cell_id'].append(rprop.label)
        table_dict['img_file'].append(img_path.name)
        table_dict['mask_file'].append(mask_path.name)

        for i, cls in enumerate(model_info['classes']):
            table_dict[f'prob_{cls}'].append(probs[i])

        table_dict['max_class'].append(class_idx)
        table_dict['max_class_name'].append(class_name)

        # draw bbox with classification
        minr, minc, maxr, maxc = rprop.bbox
        color = class_colors[class_name]
        cv2.rectangle(annotated_img, (minc, minr), (maxc, maxr), color, 1)
        text_y = max(minr - 5, 10)
        cv2.putText(annotated_img, class_name, (minc, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
    
    # --- Save Results ---
    cv2.imwrite(annotated_img_path, annotated_img)
    df = pd.DataFrame.from_dict(table_dict)
    df.to_csv(csv_path, index=False)
    
    print(f"Classified {mask.max()} regions in {img_path}.")
